In [ ]:
import sys
from pathlib import Path

# Get the root directory of the project
project_root = Path("../../KCFM_tutorial/cell_atlases").parent.parent
# project_root = Path(__file__).parent.parent
sys.path.append(str(project_root))
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from models.gene_tokenizer import GeneVocab
from models.model import MambaModel
from sklearn.metrics import classification_report, accuracy_score
import scipy.sparse

In [ ]:
# Use the same configuration parameters as training
batch_size = 256
embsize = 512
nhead = 8
d_hid = 512
nlayers = 6
dropout = 0.1
pad_token = "<pad>"
max_seq_len = 4096
input_emb_style = "continuous"
cell_emb_style = "cls"
pad_value = -2

In [ ]:
model_path = "../../model_state/cell_cls_3loss_6layer_sample_final.pth"
vocab_path = "../../models/vocab.json"

test_dir = "../../data/merlin_cxg_2023_05_15_sf-log1p/test"

output_pkl_path = "../../PKL/mamba_preds_620.pkl"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
df_features = pd.read_parquet('../../data/merlin_cxg_2023_05_15_sf-log1p/var.parquet')
vocab = GeneVocab.from_file(vocab_path)
print(f"Vocabulary loaded, size: {len(vocab)}")
feature_names = df_features['feature_name'].tolist()
print(f"Loaded {len(feature_names)} feature names")

In [ ]:
# Cell type mapping
celltype_to_id = {
    'B cell': 0,
    'Bergmann glial cell': 1,
    'CD14-low, CD16-positive monocyte': 2,    
    'CD14-positive monocyte': 3,
    'CD14-positive, CD16-negative classical monocyte': 4,
    'CD14-positive, CD16-positive monocyte': 5,
    'CD16-negative, CD56-bright natural killer cell, human': 6,
    'CD16-positive, CD56-dim natural killer cell, human': 7,
    'CD1c-positive myeloid dendritic cell': 8,
    'CD4-positive helper T cell': 9,
    'CD4-positive, alpha-beta T cell': 10,
    'CD4-positive, alpha-beta cytotoxic T cell': 11,
    'CD4-positive, alpha-beta memory T cell': 12,
    'CD8-alpha-alpha-positive, alpha-beta intraepithelial T cell': 13,
    'CD8-positive, alpha-beta T cell': 14,
    'CD8-positive, alpha-beta cytotoxic T cell': 15,
    'CD8-positive, alpha-beta memory T cell': 16,
    'IgA plasma cell': 17,
    'IgG plasma cell': 18,
    'L2/3-6 intratelencephalic projecting glutamatergic cortical neuron': 19,
    'L6b glutamatergic cortical neuron': 20,
    'Schwann cell': 21,
    'T cell': 22,
    'T follicular helper cell': 23,
    'T-helper 17 cell': 24,
    'T-helper 22 cell': 25,
    'activated CD4-positive, alpha-beta T cell': 26,
    'activated CD8-positive, alpha-beta T cell': 27,
    'alpha-beta T cell': 28,
    'alternatively activated macrophage': 29,
    'alveolar macrophage': 30,
    'alveolar type 1 fibroblast cell': 31,
    'alveolar type 2 fibroblast cell': 32,
    'amacrine cell': 33,
    'astrocyte': 34,
    'astrocyte of the cerebral cortex': 35,
    'basal cell': 36,
    'basal cell of epithelium of trachea': 37,
    'blood vessel endothelial cell': 38,
    'bronchus fibroblast of lung': 39,
    'capillary endothelial cell': 40,
    'cardiac muscle cell': 41,
    'cardiac neuron': 42,
    'caudal ganglionic eminence derived GABAergic cortical interneuron': 43,
    'central memory CD4-positive, alpha-beta T cell': 44,
    'central memory CD8-positive, alpha-beta T cell': 45,
    'central nervous system macrophage': 46,
    'chandelier pvalb GABAergic cortical interneuron': 47,
    'ciliated columnar cell of tracheobronchial tree': 48,
    'class switched memory B cell': 49,
    'classical monocyte': 50,
    'club cell': 51,
    'conventional dendritic cell': 52,
    'corticothalamic-projecting glutamatergic cortical neuron': 53,
    'dendritic cell': 54,
    'double negative T regulatory cell': 55,
    'double negative thymocyte': 56,
    'double-positive, alpha-beta thymocyte': 57,
    'effector CD8-positive, alpha-beta T cell': 58,
    'effector memory CD4-positive, alpha-beta T cell': 59,
    'effector memory CD8-positive, alpha-beta T cell': 60,
    'effector memory CD8-positive, alpha-beta T cell, terminally differentiated': 61,
    'elicited macrophage': 62,
    'endothelial cell': 63,
    'endothelial cell of artery': 64,
    'endothelial cell of lymphatic vessel': 65,
    'enteric smooth muscle cell': 66,
    'enterocyte': 67,
    'enteroendocrine cell': 68,
    'ependymal cell': 69,
    'epithelial cell of proximal tubule': 70,
    'erythroblast': 71,
    'erythrocyte': 72,
    'exhausted T cell': 73,
    'fallopian tube secretory epithelial cell': 74,
    'fibroblast of cardiac tissue': 75,
    'foveolar cell of stomach': 76,
    'gamma-delta T cell': 77,
    'glutamatergic neuron': 78,
    'goblet cell': 79,
    'granulocyte': 80,
    'granulosa cell': 81,
    'hematopoietic stem cell': 82,
    'immature B cell': 83,
    'immature innate lymphoid cell': 84,
    'inflammatory macrophage': 85,
    'innate lymphoid cell': 86,
    'intermediate monocyte': 87,
    'intestine goblet cell': 88,
    'intraepithelial lymphocyte': 89,
    'keratinocyte': 90,
    'kidney collecting duct intercalated cell': 91,
    'kidney collecting duct principal cell': 92,
    'kidney connecting tubule epithelial cell': 93,
    'kidney distal convoluted tubule epithelial cell': 94,
    'kidney interstitial fibroblast': 95,
    'kidney loop of Henle thick ascending limb epithelial cell': 96,
    'kidney loop of Henle thin ascending limb epithelial cell': 97,
    'kidney loop of Henle thin descending limb epithelial cell': 98,
    'lamp5 GABAergic cortical interneuron': 99,
    'leukocyte': 100,
    'luminal epithelial cell of mammary gland': 101,
    'lung macrophage': 102,
    'lung pericyte': 103,
    'lymphocyte': 104,
    'lymphoid lineage restricted progenitor cell': 105,
    'macrophage': 106,
    'mast cell': 107,
    'mature B cell': 108,
    'mature NK T cell': 109,
    'mature T cell': 110,
    'mature alpha-beta T cell': 111,
    'mature gamma-delta T cell': 112,
    'megakaryocyte-erythroid progenitor cell': 113,
    'memory B cell': 114,
    'memory T cell': 115,
    'mesothelial cell': 116,
    'microglial cell': 117,
    'monocyte': 118,
    'mucosal invariant T cell': 119,
    'mucous neck cell': 120,
    'myoepithelial cell of mammary gland': 121,
    'naive B cell': 122,
    'naive T cell': 123,
    'naive thymus-derived CD4-positive, alpha-beta T cell': 124,
    'naive thymus-derived CD8-positive, alpha-beta T cell': 125,
    'nasal mucosa goblet cell': 126,
    'natural killer cell': 127,
    'near-projecting glutamatergic cortical neuron': 128,
    'neuron': 129,
    'neutrophil': 130,
    'non-classical monocyte': 131,
    'oligodendrocyte': 132,
    'oligodendrocyte precursor cell': 133,
    'paneth cell': 134,
    'pericyte': 135,
    'plasma cell': 136,
    'plasmablast': 137,
    'plasmacytoid dendritic cell': 138,
    'platelet': 139,
    'precursor B cell': 140,
    'pro-B cell': 141,
    'promonocyte': 142,
    'pulmonary artery endothelial cell': 143,
    'pvalb GABAergic cortical interneuron': 144,
    'regulatory T cell': 145,
    'renal interstitial pericyte': 146,
    'respiratory basal cell': 147,
    'respiratory hillock cell': 148,
    'retina horizontal cell': 149,
    'retinal cone cell': 150,
    'retinal ganglion cell': 151,
    'retinal rod cell': 152,
    'smooth muscle cell': 153,
    'sncg GABAergic cortical interneuron': 154,
    'sst GABAergic cortical interneuron': 155,
    'tracheal goblet cell': 156,
    'tracheobronchial smooth muscle cell': 157,
    'transitional stage B cell': 158,
    'type I pneumocyte': 159,
    'type II pneumocyte': 160,
    'vascular associated smooth muscle cell': 161,
    'vein endothelial cell': 162,
    'vip GABAergic cortical interneuron': 163
}
id_to_celltype = {v: k for k, v in celltype_to_id.items()}
class_num = len(celltype_to_id)

In [ ]:
class ParquetSingleCellDataset(Dataset):
    def __init__(self, file_path):
        self.df = pd.read_parquet(file_path)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Gene expression: 1-D vector, shape [n_genes]
        x = row['X']  # already numeric (numpy or tensor)
        x = torch.FloatTensor(x)  # shape: [n_genes]

        # ground-truth label: integer IDs, e.g. 0, 1, 2, ...
        cell_type = int(row['cell_type'])

        # keep only genes with nonzero expression
        nonzero_idx = torch.nonzero(x).squeeze()  # shape: [n_nonzero]
        expr_values = x[nonzero_idx]  # nonzero values
        gene_indices = nonzero_idx  # column indices in the original matrix

        # gene name = feature_names[gene_index] (.X column order must match feature_names)
        gene_names = [feature_names[i] for i in gene_indices.tolist()]

        gene_ids = []
        filtered_expr = []
        for gene, value in zip(gene_names, expr_values.tolist()):
            if gene in vocab:  # keep only genes present in the vocab
                gene_ids.append(vocab[gene])
                filtered_expr.append(value)

        # randomly subsample if genes exceed max_seq_len - 1 (one slot for CLS)
        if len(gene_ids) > max_seq_len - 1:
            selected = np.random.choice(len(gene_ids), max_seq_len - 1, replace=False)
            gene_ids = [gene_ids[i] for i in selected]
            filtered_expr = [filtered_expr[i] for i in selected]

        # add CLS token
        gene_ids = [vocab["<cls>"]] + gene_ids
        filtered_expr = [0.0] + filtered_expr  # CLS expression set to 0.0

        # pad to fixed length max_seq_len
        padding_len = max_seq_len - len(gene_ids)
        if padding_len > 0:
            gene_ids += [vocab["<pad>"]] * padding_len
            filtered_expr += [pad_value] * padding_len

        # padding mask: True at <pad> positions
        padding_mask = [gid == vocab["<pad>"] for gid in gene_ids]

        return {
            'src': torch.LongTensor(gene_ids),
            'values': torch.FloatTensor(filtered_expr),
            'padding_mask': torch.BoolTensor(padding_mask),
            'celltype': torch.tensor(cell_type, dtype=torch.long)  # already integer IDs
        }

In [ ]:
def infer_parquet_files():
    # collect all test parquet files
    test_files = sorted(
        [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.parquet')],
        key=lambda x: int(os.path.basename(x).split('.')[1])
    )
    print(f"Found {len(test_files)}  test files")

    all_preds = []
    all_true = []

    # load model
    model = MambaModel(
        ntoken=len(vocab), d_model=512,nlayers=6,
        dropout=0.2, pad_token="<pad>", pad_value=pad_value,
        input_emb_style="continuous", cell_emb_style="cls"
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print("Model loaded")

    for file_path in tqdm(test_files, desc="Inferring"):
        dataset = ParquetSingleCellDataset(file_path)
        loader = DataLoader(dataset, batch_size=256, shuffle=False)

        for batch in loader:
            src = batch['src'].to(device)
            values = batch['values'].to(device)
            padding_mask = batch['padding_mask'].to(device)
            celltypes = batch['celltype'].to(device)

            with torch.no_grad():
                outputs = model(src=src, values=values, src_key_padding_mask=padding_mask)
                preds = outputs["cls_output"].argmax(dim=1)  # predicted class IDs

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(celltypes.cpu().numpy())

    return np.array(all_preds), np.array(all_true)

In [ ]:
# run inference, collect predictions and labels
preds, true_labels = infer_parquet_files()

# save results to pkl
results = {
    'preds': preds,
    'true_labels': true_labels,
    'celltype_to_id': celltype_to_id,
    'id_to_celltype': {v: k for k, v in celltype_to_id.items()}
}
with open(output_pkl_path, 'wb') as f:
    pickle.dump(results, f)
print(f"Inference results saved to: {output_pkl_path}")

# classification report
print("\n📊 Classification Report:")
print(classification_report(
    true_labels,
    preds,
    target_names=[id_to_celltype[i] for i in sorted(id_to_celltype.keys())]  # human-readable class names
))

# compute accuracy
accuracy = accuracy_score(true_labels, preds)
print(f"\n🎯 Overall Accuracy: {accuracy:.4f}")